# Automotive multi-echelon supply chain — work notebook

**Proposal:** [Brainstorming_Template.ipynb](Brainstorming_Template.ipynb)  
**Rubric:** [selfDefinedProject.md](selfDefinedProject.md) — **Option 2** (2 deeper Cypher + 1 GDS)

**Dataset:** Moetz et al. (2020), Mendeley Data [DOI 10.17632/pr3sdy5vp3.1](https://doi.org/10.17632/pr3sdy5vp3.1)

**Phase 1 (this notebook scaffold):** paths, Neo4j smoke test, `.xlsb` → `data_export/*.csv`, quick pandas profiles, Mermaid model, placeholders for ingest, EDA (8+), deeper questions, GDS.

Run notebooks with **working directory** `finalProject` so relative paths resolve.

## 1. Paths and imports

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

# Project root = cwd (start Jupyter from finalProject)
ROOT = Path.cwd().resolve()
WORKBOOK = ROOT / "2020_dataset_OfAutomotiveProductionNetwork.xlsb"
EXPORT_DIR = ROOT / "data_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT", ROOT)
print("WORKBOOK exists:", WORKBOOK.is_file(), WORKBOOK)
print("EXPORT_DIR:", EXPORT_DIR)

ROOT /home/themad/Documents/yeshiva/capstone/neo4j/finalProject
WORKBOOK exists: True /home/themad/Documents/yeshiva/capstone/neo4j/finalProject/2020_dataset_OfAutomotiveProductionNetwork.xlsb
EXPORT_DIR: /home/themad/Documents/yeshiva/capstone/neo4j/finalProject/data_export


## 2. Neo4j driver

Loads [`.env`](.env) via `python-dotenv`. Use the same variables as Phase 0 (`NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`).

In [4]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase, basic_auth

load_dotenv(ROOT / ".env")
uri = os.environ.get("NEO4J_URI", "bolt://localhost:7687").strip()
user = os.environ.get("NEO4J_USER", "neo4j").strip()
password = os.environ.get("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE", "").strip() or None

driver = GraphDatabase.driver(uri, auth=basic_auth(user, password), database=NEO4J_DATABASE)
driver.verify_connectivity()
print("Connected:", uri, "| database:", NEO4J_DATABASE or "(default)")


def run_query(query, parameters=None):
    """Run Cypher and return a pandas DataFrame (same style as recommender notebook)."""
    parameters = parameters or {}
    sess_kw = {"database": NEO4J_DATABASE} if NEO4J_DATABASE else {}
    with driver.session(**sess_kw) as session:
        result = session.run(query, parameters)
        return pd.DataFrame([dict(record) for record in result])


run_query("RETURN 1 AS ok")

Connected: bolt://localhost:7687 | database: finalCapstone


,ok
0,1


## 3. Extract workbook tabs → CSV (`pyxlsb`)

Expected sheets (from proposal): `products`, `nodes`, `nodes_inflow`, `arcs`, `capacity_at_arc`, `max_flow_product_per_arc`, `max_flow_group_per_arc`, `operations`, `BOM`, `demands`, `initial_inventories`, `initial_flows`.

**Neo4j `LOAD CSV`:** copy needed files into the database **import** directory; use `file:///filename.csv` in Cypher.

In [5]:
EXPECTED_SHEETS = [
    "products",
    "nodes",
    "nodes_inflow",
    "arcs",
    "capacity_at_arc",
    "max_flow_product_per_arc",
    "max_flow_group_per_arc",
    "operations",
    "BOM",
    "demands",
    "initial_inventories",
    "initial_flows",
]

if not WORKBOOK.is_file():
    print(
        "MISSING workbook. Download from Mendeley and save as:\n ",
        WORKBOOK.name,
    )
else:
    xl = pd.ExcelFile(WORKBOOK, engine="pyxlsb")
    available = set(xl.sheet_names)
    print("Sheets in file:", len(available))
    exported = []
    missing = []
    for name in EXPECTED_SHEETS:
        if name not in available:
            # try case-insensitive
            lower = {s.lower(): s for s in available}
            key = name.lower()
            if key in lower:
                name = lower[key]
            else:
                missing.append(name)
                continue
        df = pd.read_excel(WORKBOOK, sheet_name=name, engine="pyxlsb")
        out = EXPORT_DIR / f"{name}.csv"
        df.to_csv(out, index=False, encoding="utf-8")
        exported.append((name, len(df), len(df.columns)))
    for row in exported:
        print(f"  {row[0]}: {row[1]} rows × {row[2]} cols -> {row[0]}.csv")
    if missing:
        print("NOT FOUND (check exact sheet names in workbook):", missing)

Sheets in file: 12
  products: 28049 rows × 3 cols -> products.csv
  nodes: 12 rows × 1 cols -> nodes.csv
  nodes_inflow: 44 rows × 2 cols -> nodes_inflow.csv
  arcs: 11 rows × 4 cols -> arcs.csv
  capacity_at_arc: 154 rows × 4 cols -> capacity_at_arc.csv
  max_flow_product_per_arc: 365 rows × 5 cols -> max_flow_product_per_arc.csv
  max_flow_group_per_arc: 20 rows × 5 cols -> max_flow_group_per_arc.csv
  operations: 15 rows × 7 cols -> operations.csv
  BOM: 87059 rows × 3 cols -> BOM.csv
  demands: 28000 rows × 4 cols -> demands.csv
  initial_inventories: 82 rows × 6 cols -> initial_inventories.csv
  initial_flows: 117 rows × 5 cols -> initial_flows.csv


## 4. Profile exported CSVs

In [6]:
csvs = sorted(EXPORT_DIR.glob("*.csv"))
if not csvs:
    print("No CSVs in", EXPORT_DIR, "— run extraction cell above.")
else:
    for p in csvs:
        df = pd.read_csv(p, nrows=5000)
        print(f"\n=== {p.name} === shape(sample) {df.shape}")
        print(df.dtypes.head(12))
        print(df.head(3))


=== BOM.csv === shape(sample) (5000, 3)
mother                            int64
child                               str
individual_input_quantity_q_mc    int64
dtype: object
   mother child  individual_input_quantity_q_mc
0   64001   DK8                               1
1   64002   D83                               1
2   64003   DK8                               1

=== arcs.csv === shape(sample) (11, 4)
starting_node_i             str
ending_node_j               str
process_lead_time_l_ij    int64
group_g                     str
dtype: object
       starting_node_i ending_node_j  process_lead_time_l_ij group_g
0                  zp7           zp8                       0     car
1    gear-supplier_inv           zp7                       2    gear
2  engine-supplier_inv           zp7                       1  engine

=== capacity_at_arc.csv === shape(sample) (154, 4)
starting_node_i      str
ending_node_j        str
period_t           int64
capacity_c_ijt     int64
dtype: object
     star

## 5. Phase 2 — Tabular validation and data dictionary

**Goal:** Full row counts, nulls, cardinality, and cross-table checks before Phase 3 `LOAD CSV`.

**Modeling note:** `products.csv` is mostly numeric `product_p`, with a small set of alphanumeric SKUs (e.g. engine codes). `BOM.csv` uses the same string id space for `mother` and `child`. In Neo4j, `Product.productId` is a **trimmed string** for both `MERGE` endpoints. After Phase 3.3, `Product.catalogSource` is `'products_sheet'` or `'BOM_only'` (first appearance outside the sheet), and each `REQUIRES` edge has `source = 'BOM'`.

Run the **next two code cells**, then keep the dictionary table below aligned with your actual outputs. After Phase 3 ingest, use the **Graph vocabulary** subsection (below the table) when writing Cypher and GDS projections.

In [7]:
# Full-file profile: row counts, nulls, uniques (fits in memory for this dataset)
csvs = sorted(EXPORT_DIR.glob("*.csv"))
if not csvs:
    print("No CSVs in", EXPORT_DIR, "— run Section 3 extraction first.")
else:
    summaries = []
    for p in csvs:
        df = pd.read_csv(p, low_memory=False)
        na = df.isna().sum()
        na_nonzero = na[na > 0]
        print(f"\n{'=' * 70}\n{p.name}  rows={len(df):,}  cols={df.shape[1]}")
        print(df.dtypes.to_string())
        if len(na_nonzero):
            print("Null counts (nonzero):\n", na_nonzero.to_string())
        else:
            print("Null counts: none")
        print("Distinct counts:")
        for col in df.columns:
            print(f"  {col}: {df[col].nunique():,}")
        if "period_t" in df.columns:
            print(
                f"  period_t range: {df['period_t'].min()} … {df['period_t'].max()}"
            )
        summaries.append((p.name, len(df), list(df.columns)))
    print(f"\nTotal CSV files profiled: {len(summaries)}")


BOM.csv  rows=87,059  cols=3
mother                              str
child                               str
individual_input_quantity_q_mc    int64
Null counts: none
Distinct counts:
  mother: 28,049
  child: 49
  individual_input_quantity_q_mc: 4

arcs.csv  rows=11  cols=4
starting_node_i             str
ending_node_j               str
process_lead_time_l_ij    int64
group_g                     str
Null counts: none
Distinct counts:
  starting_node_i: 11
  ending_node_j: 8
  process_lead_time_l_ij: 5
  group_g: 7

capacity_at_arc.csv  rows=154  cols=4
starting_node_i      str
ending_node_j        str
period_t           int64
capacity_c_ijt     int64
Null counts: none
Distinct counts:
  starting_node_i: 11
  ending_node_j: 8
  period_t: 14
  capacity_c_ijt: 6
  period_t range: 61 … 74

demands.csv  rows=28,000  cols=4
node_n            str
product_p       int64
demand_d_npt    int64
period_t        int64
Null counts: none
Distinct counts:
  node_n: 1
  product_p: 28,000
  demand_d_np

In [8]:
# Cross-table checks: node ids and product ids vs master lists
paths = {p.stem: p for p in EXPORT_DIR.glob("*.csv")}


def _load(stem: str) -> pd.DataFrame:
    return pd.read_csv(paths[stem], low_memory=False)


def _numeric_ids(s: pd.Series) -> set[int]:
    v = pd.to_numeric(s, errors="coerce").dropna()
    return set(v.astype(int))


issues: list[str] = []

if "nodes" not in paths:
    issues.append("SKIP: nodes.csv missing")
else:
    nodes_df = _load("nodes")
    node_set = set(nodes_df["node_n"].dropna().astype(str))

    def nodes_ref_ok(stem: str, cols: list[str]) -> None:
        if stem not in paths:
            return
        df = _load(stem)
        for col in cols:
            if col not in df.columns:
                continue
            vals = set(df[col].dropna().astype(str))
            unknown = vals - node_set
            if unknown:
                sample = sorted(unknown)[:8]
                issues.append(
                    f"WARN {stem}.{col}: {len(unknown)} values not in nodes.node_n e.g. {sample}"
                )

    nodes_ref_ok("arcs", ["starting_node_i", "ending_node_j"])
    nodes_ref_ok("demands", ["node_n"])
    nodes_ref_ok("initial_inventories", ["node_n"])
    nodes_ref_ok("operations", ["node_n"])
    nodes_ref_ok("nodes_inflow", ["node_n"])
    for stem in (
        "capacity_at_arc",
        "max_flow_group_per_arc",
        "max_flow_product_per_arc",
        "initial_flows",
    ):
        nodes_ref_ok(stem, ["starting_node_i", "ending_node_j"])

if "products" not in paths:
    issues.append("SKIP: products.csv missing")
elif "BOM" in paths:
    products_df = _load("products")
    bom_df = _load("BOM")
    prod_int = _numeric_ids(products_df["product_p"])
    mothers = _numeric_ids(bom_df["mother"])
    missing_mothers = mothers - prod_int
    if missing_mothers:
        issues.append(
            f"WARN BOM.mother: {len(missing_mothers)} mothers not in products.product_p e.g. {sorted(missing_mothers)[:8]}"
        )
    prod_str = {str(int(x)) for x in prod_int}
    children = set(bom_df["child"].dropna().astype(str))
    overlap = children & prod_str
    issues.append(
        f"INFO BOM.child: {len(children)} distinct child codes; {len(overlap)} appear as stringified products.product_p (rest are part codes only in BOM)"
    )

if "demands" in paths and "products" in paths:
    ddf = _load("demands")
    pdf = _load("products")
    pset = _numeric_ids(pdf["product_p"])
    dprods = _numeric_ids(ddf["product_p"])
    missing_d = dprods - pset
    if missing_d:
        issues.append(
            f"WARN demands.product_p: {len(missing_d)} not in products e.g. {sorted(missing_d)[:8]}"
        )

if "initial_inventories" in paths and "products" in paths:
    inv = _load("initial_inventories")
    pdf = _load("products")
    prod_str = {str(x) for x in _numeric_ids(pdf["product_p"])}
    inv_p = set(inv["product_p"].dropna().astype(str))
    overlap_inv = inv_p & prod_str
    issues.append(
        f"INFO initial_inventories.product_p: {len(inv_p)} distinct symbols; {len(overlap_inv)} match stringified numeric products (others e.g. BEV-style codes — still model as :Product)"
    )

if not issues:
    print("PASS: no referential issues reported (or nothing to check).")
else:
    for line in issues:
        print(line)

INFO BOM.child: 49 distinct child codes; 0 appear as stringified products.product_p (rest are part codes only in BOM)
INFO initial_inventories.product_p: 45 distinct symbols; 0 match stringified numeric products (others e.g. BEV-style codes — still model as :Product)


### Column → graph mapping (fill / adjust after running Phase 2 cells)

| CSV file | Column | Neo4j target |
|----------|--------|----------------|
| `nodes.csv` | `node_n` | `(:Node {nodeId})` + `OEM` / `SupplierSite` / `Production` / `Inventory`, `siteKind`, `displayName` (Phase 3.4) |
| `products.csv` | `product_p` | `(:Product {productId})` — store as string |
| `products.csv` | `group_g` | `(:ProductGroup {groupName})` via `BELONGS_TO` |
| `products.csv` | `transportation_size_s` | `(:Product)` property |
| `products.csv` | ingest | `(:Product).catalogSource = 'products_sheet'` (Phase 3.3) |
| `BOM.csv` | `mother`, `child`, `individual_input_quantity_q_mc` | `(:Product)-[:REQUIRES {quantity, source:'BOM'}]->(:Product)`; `catalogSource` on endpoints coalesced to `'BOM_only'` if missing |
| `arcs.csv` | `starting_node_i`, `ending_node_j`, `process_lead_time_l_ij`, `group_g` | `(:Node)-[:SHIPS_TO {leadTime, productGroup, arcKey}]->(:Node)` |
| `nodes_inflow.csv` | `node_n`, `product_p` | `(:Node)-[:PRODUCES]->(:Product)` |
| `operations.csv` | `node_n`, groups, quantities, `alpha_nxy`, `beta_nxy` | `(:Node)-[:OUTPUTS_GROUP]` / `[:USES_INPUT]` (Phase 3.4) |
| `demands.csv` | `node_n`, `product_p`, `demand_d_npt`, `period_t` | `(:Customer)-[:ORDERS]->(:DemandFact)`; `HAS_DEMAND` / `FOR_PRODUCT` / `IN_PERIOD` / `AT_NODE` (Phase 3.4) |
| `initial_inventories.csv` | `node_n`, `product_p`, inventory fields, `period_t` | `(:Node)-[:HOLDS {…periodId}]->(:Product)` |
| `capacity_at_arc.csv` | arc ids + `period_t` + `capacity_c_ijt` | `CAPACITY_AT` with `capacity`, `periodId`, **`arcKey`** (Phase 3.5) |
| `max_flow_*` / `initial_flows` | arc/product/group + `period_t` + flow | `PLANNED_FLOW_TO` / `GROUP_FLOW_TO` / `INITIAL_FLOW_TO` with **`arcKey`** (Phase 3.5) |

**Periods:** derive `(:Period {periodId})` from distinct `period_t` values, or store `period_t` only on relationships — match your proposal in [Brainstorming_Template.ipynb](Brainstorming_Template.ipynb).

**Next:** Read **Graph vocabulary, `arcKey`, and time semantics** (subsection immediately below) for BOM vs capstone naming, `SHIPS_TO` / `arcKey`, time modeling, and the demand hub before running Phase 3 ingest.

### Graph vocabulary, `arcKey`, and time semantics

**BOM — `REQUIRES` vs `CONTAINS`:** A BOM row (mother → child) is modeled as `(mother:Product)-[:REQUIRES {quantity, source:'BOM'}]->(child:Product)`. Other notebooks may use **`CONTAINS`** for the same parent→child direction; only the **relationship type name** differs.

**Logistics — `SHIPS_TO`:** Inter-facility arcs use **`SHIPS_TO`** with `leadTime`, `productGroup`, and **`arcKey = fromNodeId + '|' + toNodeId`**. Sample Cypher elsewhere may say **`FLOWS_TO`** or **`SUPPLIES`** — in **this** notebook, query **`SHIPS_TO`**.

**Parallel arc relationships:** `CAPACITY_AT`, `PLANNED_FLOW_TO`, `GROUP_FLOW_TO`, and `INITIAL_FLOW_TO` reuse the same endpoints as `SHIPS_TO` and carry the same **`arcKey`**, plus `periodId` (and `productId` / `productGroup` where relevant). **Do not** copy `leadTime` onto those rels; join to `SHIPS_TO` on `arcKey` when you need lead time with capacity.

**Time:** Use **`periodId`** on time-varying **relationships** (`HOLDS`, capacity, flows). Use **`(:Period)`** nodes for demand via **`DemandFact`-`IN_PERIOD`-`Period`**. Opening inventory stays **`(Node)-[:HOLDS {periodId, …}]->(Product)`** only.

**Demand hub:** `(:Customer {customerId:'market'})-[:ORDERS]->(:DemandFact)` together with `HAS_DEMAND`, `FOR_PRODUCT`, `IN_PERIOD`, and `AT_NODE` for traversals.

## 6. Schema — constraints (Phase 3.2)

**Step 1 — Reset (next code cell): full wipe of this database**

- Drops **all** constraints returned by `SHOW CONSTRAINTS`.
- Runs **`MATCH (n) DETACH DELETE n`** once (removes every node and all attached relationships; no second query needed).

**Isolation:** set **`NEO4J_DATABASE`** in [`.env`](.env) to a dedicated database (e.g. `supplychain`) so you do not clear other coursework in the default `neo4j` database.

**Step 2 — Create:** run the next code cell (**Phase 3.2 step 2**) to create uniqueness on `Node.nodeId`, `Product.productId`, `ProductGroup.groupName`, `Period.periodId`, `DemandFact.demandKey`, `Customer.customerId` (singleton used when demand is linked to a market customer in Phase 3.4), then show `SHOW CONSTRAINTS` as a DataFrame.

**Prerequisite:** run **§2**. Skip Step 1 if the graph is already empty and you only need constraints.

**Drop only `supply_*` (no data loss):** run the optional **Phase 3.2 (optional) — Drop only `supply_*` constraints** code cell below to remove the six named `supply_*` constraints without deleting nodes.

In [9]:
# Phase 3.2 — Step 1: full reset (this Neo4j database only — see NEO4J_DATABASE in .env)
# Drops ALL constraints, then MATCH (n) DETACH DELETE n. Run §2 first.

def _qc_constraint(n: str) -> str:
    inner = str(n).strip().strip("`").replace("`", "``")
    return f"`{inner}`"


constraints_df = run_query(
    """
    SHOW CONSTRAINTS
    YIELD name AS constraintName
    RETURN constraintName AS name
    ORDER BY name
    """
)

n_drop = 0
if not constraints_df.empty and "name" in constraints_df.columns:
    for nm in constraints_df["name"].dropna().astype(str):
        run_query(f"DROP CONSTRAINT {_qc_constraint(nm)} IF EXISTS")
        n_drop += 1

run_query("MATCH (n) DETACH DELETE n")
print("Graph successfully cleared.")


Graph successfully cleared.


In [10]:
# Phase 3.2 (step 2) — create supply-chain uniqueness constraints


constraint_cypher = [
    "CREATE CONSTRAINT supply_node_nodeid IF NOT EXISTS FOR (n:Node) REQUIRE n.nodeId IS UNIQUE",
    "CREATE CONSTRAINT supply_product_productid IF NOT EXISTS FOR (p:Product) REQUIRE p.productId IS UNIQUE",
    "CREATE CONSTRAINT supply_productgroup_groupname IF NOT EXISTS FOR (g:ProductGroup) REQUIRE g.groupName IS UNIQUE",
    "CREATE CONSTRAINT supply_period_periodid IF NOT EXISTS FOR (t:Period) REQUIRE t.periodId IS UNIQUE",
    "CREATE CONSTRAINT supply_demandfact_demandkey IF NOT EXISTS FOR (f:DemandFact) REQUIRE f.demandKey IS UNIQUE",
    "CREATE CONSTRAINT supply_customer_customerid IF NOT EXISTS FOR (c:Customer) REQUIRE c.customerId IS UNIQUE",
]

for stmt in constraint_cypher:
    run_query(stmt)

display(
    run_query(
        """
    SHOW CONSTRAINTS
    YIELD name, type, entityType, labelsOrTypes, properties
    RETURN name, type, entityType, labelsOrTypes, properties
    ORDER BY name
    """
    )
)

,name,type,entityType,labelsOrTypes,properties
0,supply_customer_customerid,NODE_PROPERTY_UNIQUENESS,NODE,[Customer],[customerId]
1,supply_demandfact_demandkey,NODE_PROPERTY_UNIQUENESS,NODE,[DemandFact],[demandKey]
2,supply_node_nodeid,NODE_PROPERTY_UNIQUENESS,NODE,[Node],[nodeId]
3,supply_period_periodid,NODE_PROPERTY_UNIQUENESS,NODE,[Period],[periodId]
4,supply_product_productid,NODE_PROPERTY_UNIQUENESS,NODE,[Product],[productId]
5,supply_productgroup_groupname,NODE_PROPERTY_UNIQUENESS,NODE,[ProductGroup],[groupName]


## 7. Ingest — staged `LOAD CSV` (Phase 3)

### Phase 3.1 — Import path (server-side files)

`LOAD CSV ... FROM "file:///nodes.csv"` is resolved by **Neo4j on the machine running the database**, not by Jupyter. Your `data_export/` folder is only a staging area until those files live in the DB **import** directory.

1. In Neo4j Desktop: open your database → menu → **Open folder** → **Import** (or find `import` under the DBMS data directory).
2. Copy [`.env.example`](.env.example)’s `NEO4J_IMPORT_DIR` line into [`.env`](.env) and set it to that **absolute** path.
3. Run the **next code cell** to copy `data_export/*.csv` → `NEO4J_IMPORT_DIR`, or copy/symlink manually.

### Phase 3.3 — Period, products, BOM (`LOAD CSV`)

After CSVs are in the import folder, run the **Phase 3.3** code cell: `demands.csv` → distinct `(:Period)`, `products.csv` → `(:ProductGroup)` + `(:Product)` + `BELONGS_TO` and sets `Product.catalogSource = 'products_sheet'`, `BOM.csv` → `REQUIRES` with `quantity`, `source = 'BOM'`, and `catalogSource` on mother/child coalesced to `'BOM_only'` when the SKU was not already seen from `products.csv`.

Then run **Phase 3.4** (markdown + code): `nodes.csv` → `(:Node)` plus secondary labels (`OEM`, `SupplierSite`, `Production`, `Inventory`) and `siteKind` / `displayName`; `arcs.csv` → `SHIPS_TO` with `arcKey`; `nodes_inflow.csv` → **`PRODUCES`**; `demands.csv` → `Customer` + `ORDERS` + `DemandFact` + `HAS_DEMAND` / `FOR_PRODUCT` / `IN_PERIOD` / **`AT_NODE`**; `initial_inventories.csv` → `HOLDS`; `operations.csv` → **`OUTPUTS_GROUP`** + **`USES_INPUT`**.

Then run **Phase 3.5** (below): time-varying arc capacity and flows — `capacity_at_arc.csv`, `max_flow_product_per_arc.csv`, `max_flow_group_per_arc.csv`, `initial_flows.csv`.

**3.2** is §6 constraints — run before first ingest. Same `file:///…` naming for all `LOAD CSV` steps.

**Filenames on disk** must match Cypher (`demands.csv`, `products.csv`, `BOM.csv`, `nodes.csv`, `arcs.csv`, `nodes_inflow.csv`, `initial_inventories.csv`, `operations.csv`, `capacity_at_arc.csv`, `max_flow_product_per_arc.csv`, `max_flow_group_per_arc.csv`, `initial_flows.csv` — note `BOM.csv` capitalization).

In [11]:
# Phase 3.3 — LOAD CSV: Period, ProductGroup/Product/BELONGS_TO, BOM/REQUIRES
# Prerequisites: §6 constraints created; demands.csv, products.csv, BOM.csv in Neo4j import dir.
from IPython.display import display

period_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///demands.csv' AS row
WITH DISTINCT row.period_t AS pt
WHERE pt IS NOT NULL AND trim(toString(pt)) <> ''
MERGE (:Period {periodId: toInteger(pt)})
"""
run_query(period_cypher)

products_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///products.csv' AS row
WITH row,
  trim(toString(row.product_p)) AS pp,
  trim(toString(row.group_g)) AS gg
WHERE pp <> '' AND gg <> ''
MERGE (pg:ProductGroup {groupName: gg})
MERGE (pr:Product {productId: pp})
SET pr.transportationSize = coalesce(toInteger(trim(toString(row.transportation_size_s))), 0),
  pr.catalogSource = 'products_sheet'
MERGE (pr)-[:BELONGS_TO]->(pg)
"""
run_query(products_cypher)

bom_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///BOM.csv' AS row
WITH row,
  trim(toString(row.mother)) AS mraw,
  trim(toString(row.child)) AS craw
WHERE mraw <> '' AND craw <> ''
MERGE (parent:Product {productId: mraw})
MERGE (child:Product {productId: craw})
MERGE (parent)-[req:REQUIRES]->(child)
SET req.quantity = coalesce(toInteger(trim(toString(row.individual_input_quantity_q_mc))), 0),
  req.source = 'BOM',
  parent.catalogSource = coalesce(parent.catalogSource, 'BOM_only'),
  child.catalogSource = coalesce(child.catalogSource, 'BOM_only')
"""
run_query(bom_cypher)

display(run_query("MATCH (p:Period) RETURN count(p) AS periods"))
display(run_query("MATCH (pr:Product) RETURN count(pr) AS products"))
display(run_query("MATCH (pg:ProductGroup) RETURN count(pg) AS product_groups"))
display(run_query("MATCH ()-[b:BELONGS_TO]->() RETURN count(b) AS belongs_to"))
display(run_query("MATCH ()-[r:REQUIRES]->() RETURN count(r) AS requires"))
display(
    run_query(
        "MATCH (p:Product) WHERE p.catalogSource = 'products_sheet' RETURN count(p) AS products_from_sheet"
    )
)
display(
    run_query(
        "MATCH (p:Product) WHERE p.catalogSource = 'BOM_only' RETURN count(p) AS products_bom_only"
    )
)
display(run_query("MATCH ()-[r:REQUIRES]->() WHERE r.source = 'BOM' RETURN count(r) AS requires_with_bom_source"))

,periods
0,14


,products
0,28049


,product_groups
0,7


,belongs_to
0,28049


,requires
0,87059


,products_from_sheet
0,28049


,products_bom_only
0,0


,requires_with_bom_source
0,87059


### Phase 3.4 — Network nodes, arcs, facility–product, demand facts, inventory, operations

**Prerequisites:** Phase **3.3** has already run (so `Product`, `ProductGroup`, `Period` exist). Copy the CSVs listed in §7 into the Neo4j **import** directory.

| File | Graph pattern |
|------|----------------|
| `nodes.csv` | `(:Node {nodeId})` + labels `OEM` / `SupplierSite` / `Production` / `Inventory` (from id patterns); `siteKind`, `displayName` |
| `arcs.csv` | `(:Node)-[:SHIPS_TO {leadTime, productGroup, arcKey}]->(:Node)` |
| `nodes_inflow.csv` | `(:Node)-[:PRODUCES]->(:Product)` |
| `demands.csv` | `(:Customer {customerId:'market'})-[:ORDERS]->(:DemandFact)`; `(:Node)-[:HAS_DEMAND]->(f)-[:FOR_PRODUCT]->(:Product)`; `(f)-[:IN_PERIOD]->(:Period)`; `(f)-[:AT_NODE]->(:Node)` |
| `initial_inventories.csv` | `(:Node)-[:HOLDS {periodId, initialInventory, safetyStock, maxInventory}]->(:Product)` |
| `operations.csv` | `(:Node)-[:OUTPUTS_GROUP {inputGroup, …}]->(:ProductGroup)` (output); `(:Node)-[:USES_INPUT {inputGroup}]->(:ProductGroup)` (input) |

`HOLDS` keeps `periodId` on the relationship so the same node–product pair can appear across periods; demand time is modeled via `IN_PERIOD` to `(:Period)`. Run the next code cell after 3.3.

In [12]:
# Phase 3.4 — LOAD CSV: Node (+ domain labels), SHIPS_TO (+arcKey), PRODUCES, DemandFact + Customer,
# HOLDS, OUTPUTS_GROUP + USES_INPUT
# Prerequisites: Phase 3.3 complete; nodes.csv, arcs.csv, nodes_inflow.csv, demands.csv,
# initial_inventories.csv, operations.csv in Neo4j import dir.
from IPython.display import display

nodes_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
WITH trim(toString(row.node_n)) AS nn
WHERE nn <> ''
MERGE (:Node {nodeId: nn})
"""
run_query(nodes_cypher)

_node_label_queries = [
    """MATCH (n:Node)
SET n.displayName = n.nodeId,
  n.siteKind = CASE WHEN n.nodeId IN ['zp7', 'zp8'] THEN 'OEM' ELSE 'supplier' END""",
    "MATCH (n:Node) WHERE n.nodeId IN ['zp7', 'zp8'] SET n:OEM",
    "MATCH (n:Node) WHERE NOT n.nodeId IN ['zp7', 'zp8'] SET n:SupplierSite",
    "MATCH (n:Node) WHERE n.nodeId ENDS WITH '_prod' SET n:Production",
    "MATCH (n:Node) WHERE n.nodeId ENDS WITH '_inv' SET n:Inventory",
]
for _q in _node_label_queries:
    run_query(_q)

arcs_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///arcs.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  row
WHERE si <> '' AND ej <> ''
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[s:SHIPS_TO]->(b)
SET s.leadTime = coalesce(toInteger(trim(toString(row.process_lead_time_l_ij))), 0),
  s.productGroup = trim(toString(row.group_g)),
  s.arcKey = si + '|' + ej
"""
run_query(arcs_cypher)

produces_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///nodes_inflow.csv' AS row
WITH trim(toString(row.node_n)) AS nn,
  trim(toString(row.product_p)) AS pid
WHERE nn <> '' AND pid <> ''
MERGE (n:Node {nodeId: nn})
MERGE (pr:Product {productId: pid})
MERGE (n)-[:PRODUCES]->(pr)
"""
run_query(produces_cypher)

demands_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///demands.csv' AS row
WITH trim(toString(row.node_n)) AS nn,
  trim(toString(row.product_p)) AS pid,
  row
WHERE nn <> '' AND pid <> ''
WITH nn, pid, row, trim(toString(row.period_t)) AS ptc
WHERE ptc <> ''
WITH nn, pid, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (cust:Customer {customerId: 'market'})
MERGE (n:Node {nodeId: nn})
MERGE (pr:Product {productId: pid})
MERGE (t:Period {periodId: pt})
MERGE (f:DemandFact {demandKey: nn + '::' + pid + '::' + toString(pt)})
SET f.quantity = coalesce(toInteger(trim(toString(row.demand_d_npt))), 0)
MERGE (cust)-[:ORDERS]->(f)
MERGE (n)-[:HAS_DEMAND]->(f)
MERGE (f)-[:FOR_PRODUCT]->(pr)
MERGE (f)-[:IN_PERIOD]->(t)
MERGE (f)-[:AT_NODE]->(n)
"""
run_query(demands_cypher)

holds_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///initial_inventories.csv' AS row
WITH trim(toString(row.node_n)) AS nn,
  trim(toString(row.product_p)) AS pid,
  row
WHERE nn <> '' AND pid <> ''
WITH nn, pid, row, trim(toString(row.period_t)) AS ptc
WHERE ptc <> ''
WITH nn, pid, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (n:Node {nodeId: nn})
MERGE (pr:Product {productId: pid})
MERGE (n)-[h:HOLDS {periodId: pt}]->(pr)
SET h.initialInventory = coalesce(toInteger(trim(toString(row.initial_inventory_I_np0))), 0),
  h.safetyStock = coalesce(toInteger(trim(toString(row.safety_stock))), 0),
  h.maxInventory = coalesce(toInteger(trim(toString(row.max_inventory))), 0)
"""
run_query(holds_cypher)

operations_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///operations.csv' AS row
WITH trim(toString(row.node_n)) AS nn,
  trim(toString(row.input_product_group_x)) AS gx,
  trim(toString(row.output_product_group_y)) AS gy,
  row
WHERE nn <> '' AND gx <> '' AND gy <> ''
MERGE (n:Node {nodeId: nn})
MERGE (gout:ProductGroup {groupName: gy})
MERGE (gin:ProductGroup {groupName: gx})
MERGE (n)-[out:OUTPUTS_GROUP {inputGroup: gx}]->(gout)
SET out.outputGroup = gy,
  out.inputQty = coalesce(toInteger(trim(toString(row.input_quantity_in_nxy))), 0),
  out.outputQty = coalesce(toInteger(trim(toString(row.output_quantity_out_nxy))), 0),
  out.alpha = coalesce(toFloat(trim(toString(row.alpha_nxy))), 0.0),
  out.beta = coalesce(toFloat(trim(toString(row.beta_nxy))), 0.0)
MERGE (n)-[inp:USES_INPUT {inputGroup: gx}]->(gin)
"""
run_query(operations_cypher)

display(run_query("MATCH (n:Node) RETURN count(n) AS supply_nodes"))
display(run_query("MATCH (n:OEM) RETURN count(n) AS oem_nodes"))
display(run_query("MATCH ()-[s:SHIPS_TO]->() RETURN count(s) AS ships_to"))
display(run_query("MATCH ()-[p:PRODUCES]->() RETURN count(p) AS produces"))
display(run_query("MATCH (f:DemandFact) RETURN count(f) AS demand_facts"))
display(run_query("MATCH (c:Customer) RETURN count(c) AS customers"))
display(run_query("MATCH ()-[r:HAS_DEMAND]->() RETURN count(r) AS has_demand"))
display(run_query("MATCH ()-[r:ORDERS]->() RETURN count(r) AS orders"))
display(run_query("MATCH ()-[r:AT_NODE]->() RETURN count(r) AS at_node"))
display(run_query("MATCH ()-[r:FOR_PRODUCT]->() RETURN count(r) AS for_product"))
display(run_query("MATCH ()-[r:IN_PERIOD]->() RETURN count(r) AS in_period"))
display(run_query("MATCH ()-[ho:HOLDS]->() RETURN count(ho) AS holds"))
display(run_query("MATCH ()-[o:OUTPUTS_GROUP]->() RETURN count(o) AS outputs_group"))
display(run_query("MATCH ()-[u:USES_INPUT]->() RETURN count(u) AS uses_input"))

,supply_nodes
0,12


,oem_nodes
0,2


,ships_to
0,11


,produces
0,44


,demand_facts
0,28000


...

,for_product
0,28000


,in_period
0,28000


,holds
0,82


,outputs_group
0,15


,uses_input
0,15


### Phase 3.5 — Time-varying arc capacity and flows

**Prerequisites:** Phases **3.3** and **3.4** complete (`Node`, `SHIPS_TO`, `Product`, …). Copy these CSVs into the Neo4j import directory.

Relationship types follow [Brainstorming_Template.ipynb](Brainstorming_Template.ipynb) where applicable; `CAPACITY_AT` models per-period arc capacity from `capacity_at_arc`.

| File | Pattern |
|------|---------|
| `capacity_at_arc.csv` | `(:Node)-[:CAPACITY_AT {periodId, arcKey}]->(:Node)` with property `capacity` |
| `max_flow_product_per_arc.csv` | `(:Node)-[:PLANNED_FLOW_TO {periodId, productId, arcKey}]->(:Node)` with property `plannedFlow` |
| `max_flow_group_per_arc.csv` | `(:Node)-[:GROUP_FLOW_TO {periodId, productGroup, arcKey}]->(:Node)` with property `plannedFlow` |
| `initial_flows.csv` | `(:Node)-[:INITIAL_FLOW_TO {periodId, productId, arcKey}]->(:Node)` with property `initialFlow` |

`SHIPS_TO` stays the static arc from `arcs.csv`; Phase **3.5** relationships are **parallel** to it on the same node pairs, each carrying **`arcKey`** (`from|to`, same as `SHIPS_TO`) plus `periodId` (and `productId` / `productGroup` where relevant). Join static lead time to capacity with matching `arcKey` (and period). Run the next code cell after 3.4.

In [13]:
# Phase 3.5 — LOAD CSV: CAPACITY_AT, PLANNED_FLOW_TO, GROUP_FLOW_TO, INITIAL_FLOW_TO
# Prerequisites: Phases 3.3–3.4 complete; four CSVs in Neo4j import dir.
from IPython.display import display

capacity_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///capacity_at_arc.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  trim(toString(row.period_t)) AS ptc,
  row
WHERE si <> '' AND ej <> '' AND ptc <> ''
WITH si, ej, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[c:CAPACITY_AT {periodId: pt}]->(b)
SET c.capacity = coalesce(toInteger(trim(toString(row.capacity_c_ijt))), 0),
  c.arcKey = si + '|' + ej
"""
run_query(capacity_cypher)

planned_product_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///max_flow_product_per_arc.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  trim(toString(row.product_p)) AS pid,
  trim(toString(row.period_t)) AS ptc,
  row
WHERE si <> '' AND ej <> '' AND pid <> '' AND ptc <> ''
WITH si, ej, pid, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[f:PLANNED_FLOW_TO {periodId: pt, productId: pid}]->(b)
SET f.plannedFlow = coalesce(toInteger(trim(toString(row.planned_flow))), 0),
  f.arcKey = si + '|' + ej
"""
run_query(planned_product_cypher)

planned_group_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///max_flow_group_per_arc.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  trim(toString(row.group_g)) AS gn,
  trim(toString(row.period_t)) AS ptc,
  row
WHERE si <> '' AND ej <> '' AND gn <> '' AND ptc <> ''
WITH si, ej, gn, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[g:GROUP_FLOW_TO {periodId: pt, productGroup: gn}]->(b)
SET g.plannedFlow = coalesce(toInteger(trim(toString(row.planned_flow))), 0),
  g.arcKey = si + '|' + ej
"""
run_query(planned_group_cypher)

initial_flow_cypher = """
LOAD CSV WITH HEADERS FROM 'file:///initial_flows.csv' AS row
WITH trim(toString(row.starting_node_i)) AS si,
  trim(toString(row.ending_node_j)) AS ej,
  trim(toString(row.product_p)) AS pid,
  trim(toString(row.period_t)) AS ptc,
  row
WHERE si <> '' AND ej <> '' AND pid <> '' AND ptc <> ''
WITH si, ej, pid, row, toInteger(ptc) AS pt
WHERE pt IS NOT NULL
MERGE (a:Node {nodeId: si})
MERGE (b:Node {nodeId: ej})
MERGE (a)-[init:INITIAL_FLOW_TO {periodId: pt, productId: pid}]->(b)
SET init.initialFlow = coalesce(toInteger(trim(toString(row.initial_flow))), 0),
  init.arcKey = si + '|' + ej
"""
run_query(initial_flow_cypher)

display(run_query("MATCH ()-[c:CAPACITY_AT]->() RETURN count(c) AS capacity_at"))
display(run_query("MATCH ()-[f:PLANNED_FLOW_TO]->() RETURN count(f) AS planned_flow_product"))
display(run_query("MATCH ()-[g:GROUP_FLOW_TO]->() RETURN count(g) AS planned_flow_group"))
display(run_query("MATCH ()-[x:INITIAL_FLOW_TO]->() RETURN count(x) AS initial_flow_to"))
display(
    run_query(
        """
        MATCH ()-[c:CAPACITY_AT]->()
        RETURN count(*) AS capacity_rels, count(c.arcKey) AS capacity_with_arckey
        """
    )
)

,capacity_at
0,154


,planned_flow_product
0,365


,planned_flow_group
0,20


,initial_flow_to
0,117


,capacity_rels,capacity_with_arckey
0,154,154


## 8. Post-ingest checks (Phase 4)

Run **after Phases 3.3–3.5** complete (all `LOAD CSV` cells). The next code cell prints:

- **Node labels** — expect `Product` 28,049; `DemandFact` 28,000; `Period` 14; `Node` 12; `ProductGroup` 7; **`Customer` 1** (after 3.4).
- **Relationship types** — e.g. **`REQUIRES`** 87,059; **`BELONGS_TO`** 28,049; **`SHIPS_TO`** 11; **`PRODUCES`** 44; demand hub rels **28,000** each (`HAS_DEMAND`, `ORDERS`, `FOR_PRODUCT`, `IN_PERIOD`, `AT_NODE`); **`HOLDS`** 82; **`OUTPUTS_GROUP`** / **`USES_INPUT`** 15 each; **`CAPACITY_AT`** 154; **`PLANNED_FLOW_TO`** 365; **`GROUP_FLOW_TO`** 20; **`INITIAL_FLOW_TO`** 117 (row counts from the workbook CSVs).
- **`arcKey`** — `SHIPS_TO`, `CAPACITY_AT`, and parallel flow rels should have **`arcKey` populated** (`capacity_rels` = `capacity_with_arckey`, etc.).
- **Domain labels** — **`OEM`** nodes 2 (`zp7`, `zp8`).

Adjust expectations if your export differs. Use these checks after any re-ingest or constraint change.

In [14]:
from IPython.display import display

display(run_query("MATCH (n) RETURN labels(n) AS labels, count(*) AS cnt ORDER BY cnt DESC"))
display(
    run_query(
        """
        MATCH ()-[r]->()
        RETURN type(r) AS relType, count(*) AS cnt
        ORDER BY cnt DESC
        """
    )
)
display(run_query("MATCH (c:Customer) RETURN count(c) AS customers"))
display(run_query("MATCH (n:OEM) RETURN count(n) AS oem_nodes"))
display(
    run_query(
        """
        MATCH ()-[s:SHIPS_TO]->()
        RETURN count(*) AS ships_to_rels, count(s.arcKey) AS ships_to_with_arckey
        """
    )
)
display(
    run_query(
        """
        MATCH ()-[f:PLANNED_FLOW_TO]->()
        RETURN count(*) AS planned_flow_rels, count(f.arcKey) AS planned_flow_with_arckey
        """
    )
)
display(
    run_query(
        """
        MATCH ()-[x:INITIAL_FLOW_TO]->()
        RETURN count(*) AS initial_flow_rels, count(x.arcKey) AS initial_flow_with_arckey
        """
    )
)
display(
    run_query(
        """
        MATCH ()-[g:GROUP_FLOW_TO]->()
        RETURN count(*) AS group_flow_rels, count(g.arcKey) AS group_flow_with_arckey
        """
    )
)

,labels,cnt
0,[Product],28049
1,[DemandFact],28000
2,[Period],14
3,[ProductGroup],7
4,"[Node, SupplierSite, Inventory]",4
5,"[Node, SupplierSite, Production]",4
6,"[Node, OEM]",2
7,"[Node, SupplierSite]",2
8,[Customer],1


,relType,cnt
0,REQUIRES,87059
1,BELONGS_TO,28049
2,HAS_DEMAND,28000
3,ORDERS,28000
4,FOR_PRODUCT,28000
5,IN_PERIOD,28000
6,AT_NODE,28000
7,PLANNED_FLOW_TO,365
8,CAPACITY_AT,154
9,INITIAL_FLOW_TO,117


,customers
0,1


,oem_nodes
0,2


,ships_to_rels,ships_to_with_arckey
0,11,11


,planned_flow_rels,planned_flow_with_arckey
0,365,365


,initial_flow_rels,initial_flow_with_arckey
0,117,117


,group_flow_rels,group_flow_with_arckey
0,20,20


## 9. Graph EDA (Phase 5 — ≥8 questions)

Each block: **question** → Cypher → capture result in the notebook output. Replace placeholders after ingest.

### EDA 1

How many nodes, products, product groups, and periods are in the graph?

In [ ]:
eda_1_cypher = '''
// TODO: EDA question 1
RETURN 1 AS placeholder;
'''
# run_query(eda_1_cypher)
print(eda_1_cypher)

### EDA 2

Which product groups contain the most products?

In [ ]:
eda_2_cypher = '''
// TODO: EDA question 2
RETURN 1 AS placeholder;
'''
# run_query(eda_2_cypher)
print(eda_2_cypher)

### EDA 3

Which products have the most direct BOM (`REQUIRES`) dependencies?

In [ ]:
eda_3_cypher = '''
// TODO: EDA question 3
RETURN 1 AS placeholder;
'''
# run_query(eda_3_cypher)
print(eda_3_cypher)

### EDA 4

Which products have the deepest upstream component chains?

In [ ]:
eda_4_cypher = '''
// TODO: EDA question 4
RETURN 1 AS placeholder;
'''
# run_query(eda_4_cypher)
print(eda_4_cypher)

### EDA 5

Which nodes have the most inbound and outbound `SHIPS_TO` relationships?

In [ ]:
eda_5_cypher = '''
// TODO: EDA question 5
RETURN 1 AS placeholder;
'''
# run_query(eda_5_cypher)
print(eda_5_cypher)

### EDA 6

Which nodes have the most demand records?

In [ ]:
eda_6_cypher = '''
// TODO: EDA question 6
RETURN 1 AS placeholder;
'''
# run_query(eda_6_cypher)
print(eda_6_cypher)

### EDA 7

Which products appear in the most demand periods?

In [ ]:
eda_7_cypher = '''
// TODO: EDA question 7
RETURN 1 AS placeholder;
'''
# run_query(eda_7_cypher)
print(eda_7_cypher)

### EDA 8

Which nodes have the largest initial inventory or initial-flow positions?

In [ ]:
eda_8_cypher = '''
// TODO: EDA question 8
RETURN 1 AS placeholder;
'''
# run_query(eda_8_cypher)
print(eda_8_cypher)

## 10. Deeper Cypher (Phase 6 — Option 2)

Two analytical questions with traversal explanation + commentary markdown (add below each result).

### Deep 1 — BOM / supply complexity

Which finished products have the deepest or broadest upstream chains, and what does that imply for disruption exposure?

In [ ]:
deep1_cypher = '''
// TODO: deep analysis 1
RETURN 1 AS placeholder;
'''
# run_query(deep1_cypher)
print(deep1_cypher)

### Deep 2 — Arc criticality

Which arcs look most critical when combining demand exposure, lead time, and capacity limits?

In [ ]:
deep2_cypher = '''
// TODO: deep analysis 2
RETURN 1 AS placeholder;
'''
# run_query(deep2_cypher)
print(deep2_cypher)

### GDS — analytical graph layers (read before `gds.graph.project`)

Use **separate projections** or one **narrowly defined** graph — avoid one undifferentiated mix of every relationship type.

1. **Topology / logistics:** `Node` (secondary labels `OEM`, `SupplierSite`, `Production`, `Inventory`) and **`SHIPS_TO`** — e.g. betweenness on the **11-arc** facility digraph.
2. **BOM / product:** `Product`, **`REQUIRES`**, optionally **`BELONGS_TO`** / **`ProductGroup`** — multi-hop assembly and grouping.
3. **Integrated / risk:** Combine patterns in **Cypher** first; use **`gds.graph.project`** or **`gds.graph.project.cypher`** with an explicit allow-list. Example: `SHIPS_TO` + `PRODUCES` links facilities to SKUs. Do **not** project **`REQUIRES`** together with **`CAPACITY_AT`** unless you write down what a combined edge **means** for the algorithm (direction, weights, duplicates).

**Directedness:** PageRank and betweenness assume a clear **directed** or **undirected** story; `SHIPS_TO` is **directed** (`from` → `to`). Document your choice if you reverse or undirect for an algorithm.

## 11. Graph Data Science (Phase 7)

Read **GDS — analytical graph layers** (subsection above) before choosing projection node labels and relationship types.

**Projection:** e.g. `Node` + `SHIPS_TO` for logistics-only betweenness (`OEM` / `SupplierSite` are extra labels on the same nodes — project `Node` unless you use a Cypher projection). Document **directed** vs **undirected** and any **weights** (this dataset’s `SHIPS_TO` is unweighted aside from properties you choose to map).

**Algorithm:** e.g. **betweenness** on the `SHIPS_TO` subgraph — state **time complexity** implications and interpret scores (e.g. consolidation nodes like `engine-supplier_inv`).

In [ ]:
gds_placeholder = '''
// TODO: CALL gds.graph.project(...)
// TODO: CALL gds.betweenness.stream(...) or .write(...)
RETURN 1 AS gds_placeholder;
'''
# run_query(gds_placeholder)
print(gds_placeholder)